<a href="https://colab.research.google.com/github/padmini-ux02/Natural-Language-Processing-lab/blob/main/Task_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import re, requests, torch
import torch.nn as nn
from bs4 import BeautifulSoup
from tensorflow.keras.utils import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

class ChunkerModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.lstm, self.fc = nn.LSTM(128, 64, batch_first=True), nn.Linear(64, 1)

    def forward(self, x):
        return torch.sigmoid(self.fc(self.lstm(x)[0][:, -1, :]))

def fetch_text(url):
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')
    paragraphs = [p.get_text() for p in soup.find_all('p') if p.get_text().strip()]
    return re.sub(r'\s+', ' ', ' '.join(paragraphs))

def preprocess(text):
    tok = Tokenizer(num_words=5000)
    tok.fit_on_texts([text])
    return pad_sequences(tok.texts_to_sequences([text]), maxlen=100, padding='post'), tok

def segment_text(url):
    text = fetch_text(url)
    if not text.strip():
        return ["No text could be extracted from the page. Please check the URL or connection."]
    seq, tok = preprocess(text)
    return [chunk.strip() + "." for chunk in text.split('. ')[:5] if chunk.strip()]

print("Extracted Chunks:",
      *segment_text("https://en.wikipedia.org/wiki/Natural_language_processing"), sep="\n\n")

Extracted Chunks:

Natural language processing (NLP) is the processing of natural language information by a computer.

NLP is a subfield of computer science and is closely associated with artificial intelligence.

NLP is also related to information retrieval, knowledge representation, computational linguistics, and linguistics more broadly.[1] Major processing tasks in an NLP system include: speech recognition, text classification, natural language understanding, and natural language generation.

Natural language processing has its roots in the 1950s.[2] Already in 1950, Alan Turing published an article titled "Computing Machinery and Intelligence," which proposed what is now called the Turing test as a criterion of intelligence, though at the time that was not articulated as a problem separate from artificial intelligence.

The proposed test includes a task that involves the automated interpretation and generation of natural language.
